In [5]:
import pickle
import numpy as np
import pandas as pd

with open("simulation_results.pkl", "rb") as file:
    sim = pickle.load(file)

In [8]:
QUALI_POINTS = {1:10, 2:9, 3:8, 4:7, 5:6, 6:5, 7:4, 8:3, 9:2, 10:1}

def score_qualifying(quali_result, no_time_mask=None):
    # quali_result: (N, n_drivers) qualifying order before grid penalties
    # no_time_mask: (N, n_drivers) bool thats truw when driver didnt set time in Q1
    N, n_drivers = quali_result.shape
    points = np.zeros((N, n_drivers))
    for pos, p in QUALI_POINTS.items():
        points[quali_result == pos] = p
    # 11th-20th are 0, which is already the default
    if no_time_mask is not None:
        points[no_time_mask] = -5
    return points

In [23]:
RACE_RESULT_POINTS = {1:25, 2:18, 3:15, 4:12, 5:10, 6:8, 7:6, 8:4, 9:2, 10:1}

def score_race_driver(grid, finish, dnf, fl, dotd, overtakes=None):
    # grid: (N, n_drivers) starting position after grid penalties applied
    # finish: (N, n_drivers) finishing position, NaN where dnf
    # dnf: (N, n_drivers) bool
    # fl, dotd: (N, n_drivers) bool

    N, n_drivers = grid.shape
    points = np.zeros((N, n_drivers))

    finished = ~dnf
    pos_change = np.where(finished, grid - finish, 0)     # positive = gained positions
    gained = np.clip(pos_change, 0, None)
    lost = np.clip(-pos_change, 0, None)

    points += np.where(finished, gained * 1.0 - lost * 1.0, 0)

    if overtakes is not None:
        points += np.where(finished, overtakes * 1.0, 0)

    points += np.where(finished & fl, 10, 0)
    points += np.where(finished & dotd, 10, 0)

    for pos, p in RACE_RESULT_POINTS.items():
        points += np.where(finished & (finish == pos), p, 0)

    points += np.where(dnf, -20, 0)

    return points

In [15]:
def apply_boost(driver_points, boost_multiplier_by_driver):
    # boost_multiplier_by_driver: dict driver -> multiplier (ex. 2.0, 3.0) only for boosted drivers
    boosted = driver_points.copy()
    for i, d in enumerate(drivers):
        if d in boost_multiplier_by_driver:
            boosted[:, i] *= boost_multiplier_by_driver[d]
    return boosted

In [17]:
def score_constructors(
    driver_quali_points, driver_race_points, team_index, n_teams, 
    reached_q2, reached_q3, dsq_quali, dsq_race, pit_times, fastest_pit_team
):
    # driver_race_points excludes dotd bonus
    N = driver_quali_points.shape[0]
    team_quali = np.zeros((N, n_teams))
    team_race = np.zeros((N, n_teams))
    for t in range(n_teams):
        mask = team_index == t
        team_quali[:, t] = driver_quali_points[:, mask].sum(axis=1)
        team_race[:, t] = driver_race_points[:, mask].sum(axis=1)

    # Qualifying teamwork bonus
    q2_count = np.zeros((N, n_teams))
    q3_count = np.zeros((N, n_teams))
    for t in range(n_teams):
        mask = team_index == t
        q2_count[:, t] = reached_q2[:, mask].sum(axis=1)
        q3_count[:, t] = reached_q3[:, mask].sum(axis=1)

    teamwork = np.zeros((N, n_teams))
    teamwork += np.where(q2_count == 0, -1, 0)
    teamwork += np.where(q2_count == 1, 1, 0)
    teamwork += np.where(q2_count == 2, 3, 0)
    teamwork += np.where(q3_count == 1, 5, 0)
    teamwork += np.where(q3_count == 2, 10, 0)

    dsq_quali_team = np.zeros((N, n_teams))
    for t in range(n_teams):
        mask = team_idx == t
        dsq_quali_team[:, t] = dsq_quali[:, mask].sum(axis=1) * -5

    # Pit stops: bucket each stop, sum per team, plus fastest-pit-of-race bonus
    pit_points = np.zeros((N, n_teams))
    pit_points += np.where(pit_times < 2.0, 20, 0)
    pit_points += np.where((pit_times >= 2.0) * (pit_times <= 2.19), 10, 0)
    pit_points += np.where((pit_times >= 2.2) * (pit_times <= 2.49), 5, 0)
    pit_points += np.where((pit_times >= 2.5) * (pit_times <= 2.99), 2, 0)
    for row in range(N):
        pit_points[row, fastest_pit_team[row]] += 5

    dsq_race_team = np.zeros((N, n_teams))
    for t in range(n_teams):
        mask = team_index == t
        dsq_race_team[:, t] = dsq_race[:, mask].sum(axis=1) * -20

    constructor_points = team_quali + team_race + dsq_quali_team + pit_points + dsq_race_team
    return constructor_points

In [25]:
import fastf1
import pandas as pd
import numpy as np

fastf1.Cache.enable_cache('D:\\Programming\\Python\\AI\\F1 Fantasy\\cache')

session = fastf1.get_session(2026, 'baku', 'R')
session.load(telemetry=False)

results = session.results[['DriverNumber', 'Abbreviation', 'TeamName', 'GridPosition',
                             'Position', 'Status', 'Points']].copy()
results.columns = ['num', 'Driver', 'Team', 'grid', 'finish', 'status', 'f1_champ_points']
print(results)

core           INFO 	Loading data for Azerbaijan Grand Prix - Race [v3.8.3]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 22 drivers: ['63', '3', '6', '16', '12', '44', '41', '31', '87', '55', '27', '30', '81', '11', '5', '77', '43', '10', '1', '23', '14', '18']


   num Driver             Team  grid  finish    status  f1_champ_points
63  63    RUS         Mercedes   1.0     1.0  Finished             25.0
3    3    VER  Red Bull Racing   8.0     2.0  Finished             18.0
6    6    HAD  Red Bull Racing   4.0     3.0  Finished             15.0
16  16    LEC          Ferrari   2.0     4.0  Finished             12.0
12  12    ANT         Mercedes  16.0     5.0  Finished             10.0
44  44    HAM          Ferrari   6.0     6.0  Finished              8.0
41  41    LIN     Racing Bulls  15.0     7.0  Finished              6.0
31  31    OCO     Haas F1 Team  13.0     8.0  Finished              4.0
87  87    BEA     Haas F1 Team  10.0     9.0  Finished              2.0
55  55    SAI         Williams  14.0    10.0  Finished              1.0
27  27    HUL             Audi  18.0    11.0  Finished              0.0
30  30    LAW     Racing Bulls  11.0    12.0  Finished              0.0
81  81    PIA          McLaren   3.0    13.0  Finished          

In [26]:
drivers_test = results['Driver'].tolist()
D = len(drivers_test)

grid_test = results['grid'].values.reshape(1, D).astype(float)
finish_test = results['finish'].values.reshape(1, D).astype(float)
dnf_test = (results['status'] != 'Finished') & ~results['status'].str.contains('Lap', na=False)
dnf_test = dnf_test.values.reshape(1, D)
finish_test[dnf_test] = np.nan

# fastest lap — pull from lap data
laps = session.laps
fl_driver = laps.pick_fastest()['Driver']
fl_test = np.zeros((1, D), dtype=bool)
fl_test[0, drivers_test.index(fl_driver)] = True

# DOTD — not in FastF1; you'll need to hardcode this manually from the official result
dotd_driver = 'XXX'   # look up manually for your chosen race
dotd_test = np.zeros((1, D), dtype=bool)
if dotd_driver in drivers_test:
    dotd_test[0, drivers_test.index(dotd_driver)] = True

# quali result — separate session, pre-penalty official classification
quali_session = fastf1.get_session(2026, 'baku', 'Q')
quali_session.load(telemetry=False)
quali_order = quali_session.results[['Abbreviation', 'Position']].set_index('Abbreviation')['Position']
quali_result_test = np.array([[quali_order.get(d, np.nan) for d in drivers_test]])

core           INFO 	Loading data for Azerbaijan Grand Prix - Qualifying [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            IN

In [27]:
quali_pts = score_qualifying(quali_result_test)
race_pts = score_race_driver(grid_test, finish_test, dnf_test, fl_test, dotd_test, overtakes=None)

total_pts = quali_pts + race_pts
predicted = pd.Series(total_pts[0], index=drivers_test).sort_values(ascending=False)
print(predicted)

RUS    45.0
VER    27.0
HAD    23.0
ANT    21.0
LEC    19.0
LIN    14.0
HAM    13.0
OCO     9.0
SAI     7.0
HUL     7.0
PER     6.0
BEA     3.0
BOR     2.0
LAW    -1.0
PIA    -2.0
NOR   -14.0
GAS   -16.0
COL   -19.0
BOT   -20.0
ALB   -20.0
ALO   -20.0
STR   -20.0
dtype: float64
